# **Análisis Exploratorio de Datos (EDA)**
# **Airbnb Listings — Ciudad de México**

Este notebook realiza un análisis exploratorio completo sobre el dataset de Airbnb listings, orientado a **regresión** con variable objetivo: **price** (precio por noche en MXN).

**Curso**: Analítica de Datos — Universidad de Antioquia  
**Profesor**: Duván Cataño  
**Laboratorio #3**

---
# 1. Librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats

# Estilo global de gráficos
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

---
# 2. Carga de Datos

In [ ]:
# Ajustar la ruta según la ubicación del archivo en tu equipo
filepath = '../data/dataset_regresion_listings.csv'
data = pd.read_csv(filepath, sep=',')
print(f'Dataset cargado correctamente: {data.shape[0]:,} filas × {data.shape[1]} columnas')

## 2.1 Diccionario de Variables

| Variable | Tipo | Descripción |
|---|---|---|
| `id` | Numérica (ID) | Identificador único del listing |
| `name` | Categórica | Nombre del alojamiento |
| `host_id` | Numérica (ID) | Identificador del anfitrión |
| `host_name` | Categórica | Nombre del anfitrión |
| `neighbourhood_group` | Categórica | Grupo de vecindario (vacío en CDMX) |
| `neighbourhood` | Categórica | Alcaldía/delegación |
| `latitude` | Numérica | Latitud geográfica |
| `longitude` | Numérica | Longitud geográfica |
| `room_type` | Categórica | Tipo de habitación |
| `price` | **Numérica (TARGET)** | Precio por noche (MXN) |
| `minimum_nights` | Numérica | Mínimo de noches para reservar |
| `number_of_reviews` | Numérica | Total de reseñas |
| `last_review` | Fecha | Fecha de la última reseña |
| `reviews_per_month` | Numérica | Reseñas promedio por mes |
| `calculated_host_listings_count` | Numérica | Número de listings del anfitrión |
| `availability_365` | Numérica | Días disponibles en el año |
| `number_of_reviews_ltm` | Numérica | Reseñas en los últimos 12 meses |
| `license` | Categórica | Licencia del listing (vacío en CDMX) |

In [ ]:
data.head(10)

In [ ]:
data.tail(10)

In [ ]:
data.columns.tolist()

In [ ]:
data.info()

---
# 3. Identificación de Tipos de Variables

In [ ]:
# Separar variables por tipo
vars_numericas = data.select_dtypes(include=[np.number]).columns.tolist()
vars_categoricas = data.select_dtypes(include=['object']).columns.tolist()

# Excluir IDs y columnas vacías del análisis sustantivo
vars_id = ['id', 'host_id']
vars_vacias = ['neighbourhood_group', 'license']  # 100% nulos
vars_fecha = ['last_review']

vars_num_analisis = [v for v in vars_numericas if v not in vars_id + ['neighbourhood_group', 'license']]
vars_cat_analisis = [v for v in vars_categoricas if v not in vars_vacias + vars_fecha + ['name', 'host_name']]

print('--- Variables numéricas para análisis ---')
print(vars_num_analisis)
print('\n--- Variables categóricas para análisis ---')
print(vars_cat_analisis)
print('\n--- Variables excluidas (IDs, vacías, fecha, texto libre) ---')
print(vars_id + vars_vacias + vars_fecha + ['name', 'host_name'])

---
# 4. Análisis Descriptivo

In [ ]:
# Estadísticas descriptivas variables numéricas
data[vars_num_analisis].describe().round(2)

In [ ]:
# Medidas adicionales: mediana, skewness, kurtosis
desc_extra = pd.DataFrame({
    'media': data[vars_num_analisis].mean(),
    'mediana': data[vars_num_analisis].median(),
    'desv_std': data[vars_num_analisis].std(),
    'skewness': data[vars_num_analisis].skew(),
    'kurtosis': data[vars_num_analisis].kurtosis()
}).round(3)
print(desc_extra)

In [ ]:
# Variable objetivo: price
print('=== Análisis de la Variable Objetivo: price ===')
p = data['price'].dropna()
print(f'Media:             ${p.mean():>12,.2f} MXN')
print(f'Mediana:           ${p.median():>12,.2f} MXN')
print(f'Desv. Estándar:    ${p.std():>12,.2f} MXN')
print(f'Mínimo:            ${p.min():>12,.2f} MXN')
print(f'Máximo:            ${p.max():>12,.2f} MXN')
print(f'Asimetría (skew):  {p.skew():>15.3f}')
print(f'Kurtosis:          {p.kurtosis():>15.3f}')
print(f'\nObservaciones con precio:  {p.count():,} / {len(data):,} ({p.count()/len(data)*100:.1f}%)')

---
# 5. Identificación de Valores Faltantes

In [ ]:
# Resumen de valores nulos
nulos = pd.DataFrame({
    'nulos': data.isnull().sum(),
    'porcentaje': (data.isnull().sum() / len(data) * 100).round(2)
}).sort_values('porcentaje', ascending=False)
print(nulos)

# Visualización
fig, ax = plt.subplots(figsize=(10, 6))
cols_con_nulos = nulos[nulos['nulos'] > 0].index.tolist()
ax.barh(cols_con_nulos, nulos.loc[cols_con_nulos, 'porcentaje'], color='tomato', edgecolor='black')
ax.set_xlabel('Porcentaje de valores faltantes (%)')
ax.set_title('Valores Faltantes por Variable')
for i, (col, row) in enumerate(nulos[nulos['nulos'] > 0].iterrows()):
    ax.text(row['porcentaje'] + 0.5, i, f"{row['porcentaje']}%", va='center', fontsize=9)
plt.tight_layout()
plt.show()

**Observaciones:**
- `neighbourhood_group` y `license`: **100% nulos** → columnas vacías, se eliminan del análisis.
- `price`: ~12.9% nulos → variable objetivo, requiere imputación o eliminación según el modelo.
- `last_review` y `reviews_per_month`: ~12.6% nulos → listings sin ninguna reseña registrada (correlacionados).
- `host_name`: <0.1% nulos → imputación simple.

---
# 6. Distribuciones Univariadas (Histogramas)

In [ ]:
# Histogramas de variables numéricas (sin price aún — se ve después con detalle)
vars_hist = ['price', 'minimum_nights', 'number_of_reviews',
             'reviews_per_month', 'calculated_host_listings_count',
             'availability_365', 'number_of_reviews_ltm']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.ravel()

for i, var in enumerate(vars_hist):
    col = data[var].dropna()
    axes[i].hist(col, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribución: {var}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Frecuencia')
    axes[i].axvline(col.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Media: {col.mean():.1f}')
    axes[i].axvline(col.median(), color='orange', linestyle='-', linewidth=1.5, label=f'Mediana: {col.median():.1f}')
    axes[i].legend(fontsize=8)

# Histograma geográfico (latitude y longitude)
axes[7].hist(data['latitude'], bins=50, color='mediumseagreen', edgecolor='white', alpha=0.8)
axes[7].set_title('Distribución: latitude', fontsize=11, fontweight='bold')
axes[7].set_xlabel('latitude')
axes[8].hist(data['longitude'], bins=50, color='mediumpurple', edgecolor='white', alpha=0.8)
axes[8].set_title('Distribución: longitude', fontsize=11, fontweight='bold')
axes[8].set_xlabel('longitude')

plt.suptitle('Distribuciones Univariadas — Airbnb Listings CDMX', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Distribución del precio filtrada (sin outliers extremos) para mejor visualización
price_filtrado = data['price'][data['price'] <= 10000].dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma precio completo
axes[0].hist(data['price'].dropna(), bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribución Precio (completa, con outliers)', fontweight='bold')
axes[0].set_xlabel('Precio por noche (MXN)')
axes[0].set_ylabel('Frecuencia')

# Histograma precio recortado
axes[1].hist(price_filtrado, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(price_filtrado.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Media: ${price_filtrado.mean():,.0f}')
axes[1].axvline(price_filtrado.median(), color='orange', linestyle='-', linewidth=1.5, label=f'Mediana: ${price_filtrado.median():,.0f}')
axes[1].set_title('Distribución Precio (≤ $10,000 MXN/noche)', fontweight='bold')
axes[1].set_xlabel('Precio por noche (MXN)')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.suptitle('Variable Objetivo: price', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Nota: La distribución del precio es fuertemente sesgada a la derecha (skew = {data["price"].skew():.2f}).')
print(f'La media (${data["price"].mean():,.0f}) es mucho mayor que la mediana (${data["price"].median():,.0f}) debido a los valores extremos.')

In [ ]:
# Distribución de variables categóricas clave
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Room type
rt_counts = data['room_type'].value_counts()
axes[0].bar(rt_counts.index, rt_counts.values, color=sns.color_palette('husl', len(rt_counts)), edgecolor='black')
axes[0].set_title('Distribución por Tipo de Habitación', fontweight='bold')
axes[0].set_xlabel('room_type')
axes[0].set_ylabel('Cantidad de listings')
for i, (v) in enumerate(rt_counts.values):
    axes[0].text(i, v + 100, f'{v:,}\n({v/len(data)*100:.1f}%)', ha='center', fontsize=9)

# Neighbourhood (top 10)
top_nb = data['neighbourhood'].value_counts().head(10)
axes[1].barh(top_nb.index[::-1], top_nb.values[::-1], color='steelblue', edgecolor='black')
axes[1].set_title('Top 10 Alcaldías por Número de Listings', fontweight='bold')
axes[1].set_xlabel('Cantidad de listings')
for i, v in enumerate(top_nb.values[::-1]):
    axes[1].text(v + 50, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
# 7. Identificación de Valores Atípicos (Outliers)

In [ ]:
# Método IQR para detección de outliers
def calcular_outliers_iqr(serie):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = serie[(serie < lower) | (serie > upper)]
    return {'Q1': Q1, 'Q3': Q3, 'IQR': IQR, 'lower': lower, 'upper': upper,
            'n_outliers': len(outliers), 'pct_outliers': len(outliers)/len(serie)*100}

vars_outlier = ['price', 'minimum_nights', 'number_of_reviews',
                'reviews_per_month', 'calculated_host_listings_count',
                'availability_365', 'number_of_reviews_ltm']

resumen_outliers = pd.DataFrame({
    v: calcular_outliers_iqr(data[v].dropna()) for v in vars_outlier
}).T.round(2)
print(resumen_outliers)

In [ ]:
# Boxplots para visualizar outliers
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for i, var in enumerate(vars_outlier):
    axes[i].boxplot(data[var].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(f'{var}', fontweight='bold')
    axes[i].set_ylabel('Valor')

axes[7].axis('off')
plt.suptitle('Boxplots — Detección de Outliers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Casos extremos de precio
print('Top 10 listings más caros:')
print(data[['name', 'neighbourhood', 'room_type', 'price', 'minimum_nights']]
      .nlargest(10, 'price').to_string(index=False))

print('\nListings con precio < $200 MXN/noche:')
print(data[data['price'] < 200][['name', 'neighbourhood', 'room_type', 'price']].head(10).to_string(index=False))

---
# 8. Análisis de Correlación

In [ ]:
# Matriz de correlación de Pearson
vars_corr = ['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month',
             'calculated_host_listings_count', 'availability_365',
             'number_of_reviews_ltm', 'latitude', 'longitude']

corr_matrix = data[vars_corr].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlación de Pearson — Airbnb Listings CDMX', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlación específica con la variable objetivo: price
corr_con_price = corr_matrix['price'].drop('price').sort_values(key=abs, ascending=False)
print('Correlación de Pearson con price (ordenado por magnitud):')
print(corr_con_price.round(4))

# Visualización
plt.figure(figsize=(9, 5))
colors = ['tomato' if v < 0 else 'steelblue' for v in corr_con_price]
plt.barh(corr_con_price.index, corr_con_price.values, color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Correlación de Pearson')
plt.title('Correlación de Variables Numéricas con Price', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlación de Spearman (más robusta ante outliers y no linealidad)
corr_spearman = data[vars_corr].corr(method='spearman')['price'].drop('price').sort_values(key=abs, ascending=False)
print('Correlación de Spearman con price:')
print(corr_spearman.round(4))

# Comparativa Pearson vs Spearman
comp = pd.DataFrame({'Pearson': corr_con_price, 'Spearman': corr_spearman}).round(4)
print('\nComparativa Pearson vs Spearman:')
print(comp)

---
# 9. Análisis de Relaciones entre Variables — Scatter Plots

In [ ]:
# Scatter: minimum_nights vs price
data_plot = data[data['price'] <= 10000].dropna(subset=['price', 'minimum_nights'])

plt.figure(figsize=(10, 6))
plt.scatter(data_plot['minimum_nights'], data_plot['price'],
            alpha=0.3, color='steelblue', s=15, edgecolors='none')
plt.xlabel('Mínimo de Noches')
plt.ylabel('Precio por Noche (MXN)')
plt.title('Mínimo de Noches vs Precio por Noche', fontweight='bold')
plt.xlim(0, 30)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: number_of_reviews vs price
plt.figure(figsize=(10, 6))
plt.scatter(data_plot['number_of_reviews'], data_plot['price'],
            alpha=0.3, color='mediumseagreen', s=15, edgecolors='none')
plt.xlabel('Número de Reseñas')
plt.ylabel('Precio por Noche (MXN)')
plt.title('Número de Reseñas vs Precio por Noche', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: availability_365 vs price
plt.figure(figsize=(10, 6))
plt.scatter(data_plot['availability_365'], data_plot['price'],
            alpha=0.3, color='mediumpurple', s=15, edgecolors='none')
plt.xlabel('Disponibilidad (días en el año)')
plt.ylabel('Precio por Noche (MXN)')
plt.title('Disponibilidad 365 vs Precio por Noche', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: ubicación geográfica coloreada por precio
data_geo = data.dropna(subset=['price', 'latitude', 'longitude'])
data_geo = data_geo[data_geo['price'] <= 5000]  # filtrar outliers extremos

plt.figure(figsize=(12, 8))
sc = plt.scatter(data_geo['longitude'], data_geo['latitude'],
                 c=data_geo['price'], cmap='YlOrRd', alpha=0.5, s=10, edgecolors='none')
plt.colorbar(sc, label='Precio por Noche (MXN)')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.title('Distribución Geográfica de Listings por Precio (CDMX)', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# 10. Análisis por Variables Categóricas

In [ ]:
# Precio por tipo de habitación
print('Precio promedio y mediano por tipo de habitación:')
print(data.groupby('room_type')['price']
      .agg(['count', 'mean', 'median', 'std'])
      .rename(columns={'count': 'listings', 'mean': 'media', 'median': 'mediana', 'std': 'desv_std'})
      .round(0))

In [ ]:
# Boxplot: precio por tipo de habitación
data_box = data[data['price'] <= 8000].dropna(subset=['price'])

plt.figure(figsize=(12, 6))
room_types = data_box['room_type'].unique()
datos_por_tipo = [data_box[data_box['room_type'] == rt]['price'].values for rt in room_types]
plt.boxplot(datos_por_tipo, labels=room_types, patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.7),
            medianprops=dict(color='red', linewidth=2))
plt.xlabel('Tipo de Habitación')
plt.ylabel('Precio por Noche (MXN)')
plt.title('Distribución de Precio por Tipo de Habitación (price ≤ $8,000)', fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Precio promedio por alcaldía (top 10)
precio_alcaldia = data.groupby('neighbourhood')['price'].agg(['mean', 'median', 'count'])
precio_alcaldia = precio_alcaldia[precio_alcaldia['count'] >= 50].sort_values('mean', ascending=False)
print('Precio promedio por alcaldía (mín. 50 listings):')
print(precio_alcaldia.round(0).to_string())

top10 = precio_alcaldia.head(10)
plt.figure(figsize=(12, 6))
x = range(len(top10))
width = 0.35
plt.bar([i - width/2 for i in x], top10['mean'], width, label='Media', color='steelblue', edgecolor='black')
plt.bar([i + width/2 for i in x], top10['median'], width, label='Mediana', color='tomato', edgecolor='black')
plt.xticks(x, top10.index, rotation=30, ha='right')
plt.xlabel('Alcaldía')
plt.ylabel('Precio por Noche (MXN)')
plt.title('Top 10 Alcaldías: Media y Mediana del Precio', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
# 11. Pruebas Estadísticas de Asociación

In [ ]:
# ANOVA: ¿Hay diferencia significativa de precio entre room_types?
from scipy.stats import f_oneway, kruskal

grupos_rt = [data[data['room_type'] == rt]['price'].dropna() for rt in data['room_type'].unique()]
f_stat, p_anova = f_oneway(*grupos_rt)
h_stat, p_kruskal = kruskal(*grupos_rt)

print('=== Prueba ANOVA — Precio por Tipo de Habitación ===')
print(f'F-estadístico: {f_stat:.4f}')
print(f'p-valor:       {p_anova:.6f}')
print(f'Conclusión:    {"Se rechaza H0 → hay diferencias significativas de precio entre tipos" if p_anova < 0.05 else "No se rechaza H0"}')

print('\n=== Prueba Kruskal-Wallis (no paramétrica) ===')
print(f'H-estadístico: {h_stat:.4f}')
print(f'p-valor:       {p_kruskal:.6f}')
print(f'Conclusión:    {"Se rechaza H0 → distribuciones de precio difieren entre tipos" if p_kruskal < 0.05 else "No se rechaza H0"}')

In [ ]:
# Pearson y Spearman — Correlaciones con precio y sus p-valores
print('Correlaciones con price (Pearson y Spearman) + p-valor:')
print(f'{"Variable":<35} {"Pearson r":>10} {"p Pearson":>12} {"Spearman r":>12} {"p Spearman":>12}')
print('-'*85)

vars_test = ['minimum_nights', 'number_of_reviews', 'reviews_per_month',
             'calculated_host_listings_count', 'availability_365',
             'number_of_reviews_ltm', 'latitude', 'longitude']

for var in vars_test:
    df_pair = data[['price', var]].dropna()
    r_p, pv_p = stats.pearsonr(df_pair['price'], df_pair[var])
    r_s, pv_s = stats.spearmanr(df_pair['price'], df_pair[var])
    sig_p = '***' if pv_p < 0.001 else ('**' if pv_p < 0.01 else ('*' if pv_p < 0.05 else ''))
    sig_s = '***' if pv_s < 0.001 else ('**' if pv_s < 0.01 else ('*' if pv_s < 0.05 else ''))
    print(f'{var:<35} {r_p:>10.4f} {pv_p:>12.4e}{sig_p:<3} {r_s:>12.4f} {pv_s:>12.4e}{sig_s}')

print('\nSignificancia: *** p<0.001  ** p<0.01  * p<0.05')

---
# 12. Análisis de Anfitriones

In [ ]:
# Anfitriones con más listings
print('Top 10 anfitriones por número de listings:')
top_hosts = data.groupby(['host_id', 'host_name']).agg(
    n_listings=('id', 'count'),
    precio_medio=('price', 'mean')
).sort_values('n_listings', ascending=False).head(10).round(0)
print(top_hosts.to_string())

In [ ]:
# Distribución: cantidad de listings por anfitrión
listings_por_host = data.groupby('host_id')['id'].count()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(listings_por_host, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribución: Listings por Anfitrión', fontweight='bold')
axes[0].set_xlabel('Número de Listings')
axes[0].set_ylabel('Frecuencia')
axes[0].set_xlim(0, 50)

# Scatter: listings del anfitrión vs precio
data_plot2 = data[data['price'] <= 8000].dropna(subset=['price'])
axes[1].scatter(data_plot2['calculated_host_listings_count'], data_plot2['price'],
                alpha=0.2, s=10, color='mediumpurple', edgecolors='none')
axes[1].set_xlabel('Listings del Anfitrión')
axes[1].set_ylabel('Precio por Noche (MXN)')
axes[1].set_title('Listings del Anfitrión vs Precio', fontweight='bold')
axes[1].set_xlim(0, 80)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# 13. Resumen y Conclusiones del EDA

In [ ]:
print('=' * 65)
print('RESUMEN EDA — AIRBNB LISTINGS CDMX')
print('=' * 65)

print(f'''
1. TAMAÑO Y ESTRUCTURA
   - {len(data):,} listings, 18 variables.
   - Variables numéricas: {len(vars_num_analisis)}
   - Variables categóricas útiles: {len(vars_cat_analisis)}
   - Columnas vacías (100% nulos): neighbourhood_group, license → eliminar.

2. VARIABLE OBJETIVO: price
   - Media: ${data['price'].mean():,.0f} MXN | Mediana: ${data['price'].median():,.0f} MXN
   - Fuertemente sesgada a la derecha (skew = {data['price'].skew():.2f}).
   - Se recomienda transformación logarítmica (log(price)) para modelado.
   - {data['price'].isnull().sum():,} registros sin precio ({data['price'].isnull().mean()*100:.1f}%).

3. VALORES ATÍPICOS
   - 1,768 registros con precio fuera del rango IQR (> $3,063 MXN/noche).
   - Precios máximos: hasta $900,000 MXN/noche (probablemente errores).
   - minimum_nights también tiene outliers extremos (hasta 1,125 noches).

4. DATOS FALTANTES
   - price: 12.9% | last_review y reviews_per_month: 12.6% (correlacionados).
   - Listings sin ninguna reseña → reviews_per_month es NaN naturalmente.

5. CORRELACIÓN CON price
   - Ninguna variable numérica tiene correlación fuerte con price.
   - Las correlaciones de Pearson son bajas (<|0.05|), indicando relaciones no lineales.
   - Spearman muestra valores ligeramente mayores pero igualmente bajos.
   - El tipo de habitación (room_type) y la alcaldía tienen mayor impacto explicativo.

6. CATEGORÍAS IMPORTANTES
   - room_type: 65.4% son "Entire home/apt" con precio medio de ~$1,930 MXN.
   - Alcaldía con mayor precio promedio: Tlalpan ($2,493), Cuajimalpa ($2,151), Cuauhtémoc ($2,126).
   - Cuauhtémoc concentra el 46.3% de todos los listings.

7. RECOMENDACIONES PARA MODELADO
   - Aplicar log(price) para normalizar la variable objetivo.
   - Imputar price con KNN o IterativeImputer (más robusto que media/mediana).
   - Codificar room_type y neighbourhood con one-hot o target encoding.
   - Considerar eliminar outliers extremos de price (>$15,000) antes de modelar.
   - Latitude y longitude pueden incluirse como features geoespaciales.
''')